# 01 — Data Acquisition

Pulls the Kepler Objects of Interest (KOI) cumulative table from the
[NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu) via its
TAP (Table Access Protocol) API.

Each row in the KOI table is a signal that the Kepler telescope flagged as a
possible transiting planet, labeled by NASA as **CONFIRMED**, **CANDIDATE**,
or **FALSE POSITIVE** — the label this project will try to predict.

Raw data is saved to `data/raw/` (not committed to git).

In [9]:
import requests
import pandas as pd
import io
from pathlib import Path


## Fetch the KOI table via TAP

TAP (Table Access Protocol) lets us query the archive's database directly
with SQL over HTTP — no manual downloads, fully reproducible.

In [10]:
TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
# Pull ALL columns deliberately: acquisition archives raw reality as-is.
# Column filtering (incl. removing leakage columns) happens in the
# cleaning notebook, where those decisions are visible and documented.
params = {
    "query": "select * from cumulative",    # 'cumulative' = NASA's KOI table
    "format": "csv",                        # parses straight into pandas
}

response = requests.get(TAP_URL, params=params)
print(response.status_code)

200


In [11]:
# response.text is one giant CSV string in memory.
# StringIO wraps it in a file-like object so read_csv can parse it.
df = pd.read_csv(io.StringIO(response.text))

print(df.shape)
df.head()


df["koi_disposition"].value_counts()

(9564, 153)


koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2747
CANDIDATE         1978
Name: count, dtype: int64

In [12]:
# Save the raw pull to disk. ../ because this notebook lives in notebooks/
# and data/ belongs at the project root. Folder is gitignored — the data
# is recreated by rerunning this notebook, not committed.
out_dir = Path("../data/raw")
out_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(out_dir / "koi_cumulative.csv", index=False)
print(f"Saved {len(df):,} rows to {out_dir / 'koi_cumulative.csv'}")

Saved 9,564 rows to ../data/raw/koi_cumulative.csv


## Summary

- Retrieved **9,564 signals × 153 columns** from the KOI cumulative table.
- Class balance: 4,839 FALSE POSITIVE / 2,747 CONFIRMED / 1,978 CANDIDATE —
  about half the dataset is false alarms, so accuracy alone will be a
  misleading metric later.
- The API returned 153 columns vs ~50 shown on the website — includes error
  bounds and vetting outputs. Several columns (koi_score, koi_pdisposition,
  koi_fpflag_*) contain NASA's own vetting verdicts and must be dropped
  before modeling to avoid leakage.
- Raw data saved to `data/raw/koi_cumulative.csv`.